In [2]:
with open(".gitignore", "w") as f:
    f.write(".ipynb_checkpoints/\n__pycache__/\n*.pyc\n.DS_Store\nvenv/\n")

print(".gitignore created successfully")

.gitignore created successfully


In [1]:
import os
print(os.listdir("."))

['.gitignore', '.ipynb_checkpoints', 'app.py', 'data', 'requirements.txt', 'train_model.ipynb']


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# Load the same exported data used in Power BI
df = pd.read_csv("data/boat_scored_orders.csv")
df.head()

,SalesID,CustomerTier,CustomerSegment,CityTier,ChannelType,PaymentMethod,ProductCategory,OrderHour,Quantity,GrossMRPValue,DiscountPct,IsFestivalPeriod,IsWeekend,HeroFlag,OrderStatus,target,prediction,risk_score,risk_category
0,1,Tier-2,Student,Tier-2,Offline Retail,COD,Chargers & Cables,20,1,1140.0,0.3689,1,1,1,Delivered,0,0.0,0.0262,Low Risk
1,2,Tier-1,Working Professional,Tier-1,Offline Retail,Credit/Debit Card,Smart Watches,17,2,21500.0,0.4150,1,1,1,Delivered,0,0.0,0.0751,Low Risk
2,3,Tier-3,Gift Buyer,Tier-3,Online Marketplace,UPI,Smart Watches,16,1,10750.0,0.5633,1,1,1,Delivered,0,0.0,0.0848,Low Risk
3,4,Tier-1,Family Buyer,Tier-1,Online Marketplace,Wallet,Neckbands,17,1,2010.0,0.6133,1,1,1,Delivered,0,0.0,0.0593,Low Risk
4,5,Tier-1,Student,Tier-1,Online Marketplace,Wallet,Neckbands,8,1,1160.0,0.4518,1,1,1,Delivered,0,0.0,0.0625,Low Risk


In [3]:
# Same feature split as your Databricks Cell 22 (numeric vs categorical)
numeric_features = [
    "OrderHour", "Quantity", "GrossMRPValue", "DiscountPct",
    "IsFestivalPeriod", "IsWeekend", "HeroFlag"
]
categorical_features = [
    "CustomerTier", "CustomerSegment", "CityTier",
    "ChannelType", "PaymentMethod", "ProductCategory"
]

X = df[numeric_features + categorical_features]
y = df["target"]

# Same 80/20 split logic as Databricks
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train rows: {len(X_train):,}, Test rows: {len(X_test):,}")

Train rows: 780,000, Test rows: 195,000


In [4]:
# Preprocessing: one-hot encode categoricals, pass numeric through unchanged
# (equivalent to your StringIndexer + OneHotEncoder pipeline in PySpark)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

# Same algorithm family as your Databricks "Challenger" model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ))
])

print("Training model...")
model.fit(X_train, y_train)
print("Training complete!")

Training model...
Training complete!


In [5]:
# Evaluate — same metrics philosophy as Databricks Part 9
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC: 0.5806

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97    182764
           1       0.00      0.00      0.00     12236

    accuracy                           0.94    195000
   macro avg       0.47      0.50      0.48    195000
weighted avg       0.88      0.94      0.91    195000



In [6]:
# Save the trained pipeline (preprocessing + model, bundled together)
joblib.dump(model, "risk_model.pkl")
print("Model saved to risk_model.pkl")

# Save the distinct values for each categorical field —
# the Streamlit app needs these to build dropdown menus
categorical_options = {
    col: sorted(df[col].dropna().unique().tolist())
    for col in categorical_features
}
joblib.dump(categorical_options, "categorical_options.pkl")
print("Categorical options saved to categorical_options.pkl")

Model saved to risk_model.pkl
Categorical options saved to categorical_options.pkl


In [8]:
import os
print(os.getcwd())

C:\Users\shita\boat-risk-app


In [9]:
 df = pd.read_csv('data/boAt_scored_orders.csv'); df.to_csv('data/boAt_scored_orders.csv.gz', index=False, compression='gzip')

In [10]:
import os
size_mb = os.path.getsize("data/boAt_scored_orders.csv.gz") / (1024 * 1024)
print(f"{size_mb:.2f} MB")

13.70 MB
